In [ ]:
import os, sys
import torch

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))
import config

from dingo.core.posterior_models.build_model import build_model_from_kwargs
from dingo.gw.inference.gw_samplers import GWSampler, GWSamplerGNPE
import dingo.gw.injection as injection
from dingo.gw.noise.asd_dataset import ASDDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# Released eccentric aligned-spin net (high-mass H1L1). The `_time` net is the
# time-marginalized initialization the GNPE sampler is seeded from.
EHM = os.path.join(config.NETWORKS_DIR, "SEOBNRv5EHM")
MODEL_PATH = os.path.join(EHM, "O4_HL_0", "model_latest.pt")
MODEL_INIT_PATH = os.path.join(EHM, "O4_HL_0_time", "model_latest.pt")

main_pm = build_model_from_kwargs(filename=MODEL_PATH, device=DEVICE, load_training_info=False)
init_pm = build_model_from_kwargs(filename=MODEL_INIT_PATH, device=DEVICE, load_training_info=False)
print("approximant:", main_pm.metadata["dataset_settings"]["waveform_generator"]["approximant"])
print("inference parameters:", main_pm.metadata["train_settings"]["data"]["inference_parameters"])

## Part 1 — Analyze an injection

`Injection.from_posterior_model_metadata` builds a signal generator whose waveform
settings, domain and prior all match what the network was trained on. We colour the
noise with the **fiducial ASD the network was trained on** (read straight from the
model metadata), draw a truth from the training prior, and inject.

In [ ]:
inj = injection.Injection.from_posterior_model_metadata(main_pm.metadata)

# Use the fiducial ASD the net was trained on (one fixed realization, index 0).
asd_path = main_pm.metadata["train_settings"]["training"]["stage_0"]["asd_dataset_path"]
asd_dataset = ASDDataset(file_name=asd_path)
inj.asd = {det: asds[0] for det, asds in asd_dataset.asds.items()}

print("injection prior (intrinsic + extrinsic):")
print(inj.prior)

In [ ]:
# Draw a truth from the training prior (guarantees every parameter is in bounds),
# then nudge a few to get a confident, clearly-eccentric injection.
torch.manual_seed(0)
import numpy as np; np.random.seed(0)

theta = {k: float(v) for k, v in inj.prior.sample().items()}
theta["eccentricity"] = 0.15           # set within the prior's eccentricity range
theta["luminosity_distance"] = 600.0   # Mpc -> loud signal
theta["geocent_time"] = 0.0            # offset from the model reference time

print("injected truth:")
for k, v in theta.items():
    print(f"  {k:<22}{v:.4f}")

strain_data = inj.injection(theta)   # signal + coloured Gaussian noise (not whitened)

In [ ]:
# GNPE sampler: the init net provides time-shift proxies, then `num_iterations` Gibbs
# steps refine them. Conditioning is just `sampler.context = strain_data`.
init_sampler = GWSampler(model=init_pm)
sampler = GWSamplerGNPE(model=main_pm, init_sampler=init_sampler, num_iterations=30)
sampler.context = strain_data
sampler.run_sampler(num_samples=50_000, batch_size=10_000)

result = sampler.to_result()
result.plot_corner(filename="injection_corner.png")
print(result.samples[["chirp_mass", "mass_ratio", "eccentricity", "chi_1", "chi_2"]].describe())

## Part 2 — Analyze a real event quickly (no importance sampling)

**GW230706_104333** (O4a). We download public GWOSC strain around the trigger, estimate the
PSD, and run the same GNPE sampler. This is the **proposal** posterior straight from the
flow — fast, but *not* importance-sampled, so the samples are not yet reweighted to the
exact likelihood. (Importance sampling is what `dingo_pipe` adds in Part 3.)

Requires network access to GWOSC; the strain for the requested GPS time must be public.

In [ ]:
from dingo.gw.data.data_preparation import get_event_data_and_domain, parse_settings_for_raw_data

TIME_EVENT = 1372675431.156993   # GW230706_104333 trigger (GPS)
TIME_PSD = 1024                  # s of data for the PSD estimate
TIME_BUFFER = 2.0                # s of post-trigger data

event_data, _ = get_event_data_and_domain(main_pm.metadata, TIME_EVENT, TIME_PSD, TIME_BUFFER)

# Build the event metadata in the shape the sampler expects (mirrors dingo's
# inference pipeline) so the reference-time / RA correction is applied correctly.
event_metadata = parse_settings_for_raw_data(main_pm.metadata, TIME_PSD, TIME_BUFFER)
event_metadata["time_event"] = TIME_EVENT
event_metadata["psd_duration"] = event_metadata.pop("time_psd")
_window = event_metadata.pop("window")
event_metadata["T"] = _window["T"]
del event_metadata["time_segment"]
event_metadata["window_type"] = _window["type"]
event_metadata["roll_off"] = _window["roll_off"]
event_metadata["f_min"] = main_pm.metadata["dataset_settings"]["domain"]["f_min"]
event_metadata["f_max"] = main_pm.metadata["dataset_settings"]["domain"]["f_max"]

In [ ]:
init_sampler = GWSampler(model=init_pm)
event_sampler = GWSamplerGNPE(model=main_pm, init_sampler=init_sampler, num_iterations=30)
event_sampler.context = event_data
event_sampler.event_metadata = event_metadata
event_sampler.run_sampler(num_samples=50_000, batch_size=10_000)

event_result = event_sampler.to_result()
event_result.plot_corner(filename="GW230706_104333_corner.png")
print(event_result.samples[["chirp_mass", "mass_ratio", "eccentricity"]].describe())

## Part 3 — Set up a `dingo_pipe` run

For production you don't drive the sampler by hand — you write an ini and run
`dingo_pipe`, which generates the data, runs the GNPE flow, **and importance-samples**
the result against the full likelihood (the step Part 2 skipped). The cell below writes
a run folder with an ini adapted from our real GW230706_104333 analysis: same trigger and
prior, but pointed at the released eccentric net and at public GWOSC channels so it is
runnable out of the box.

In [ ]:
run_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "dingo_pipe_GW230706_104333"))
os.makedirs(run_dir, exist_ok=True)
ini_path = os.path.join(run_dir, "GW230706_104333.ini")

ini = f'''################################################################################
## Job submission
################################################################################
local = True                 ; run locally; set False to submit to HTCondor
local-generation = True
accounting = ligo.dev.o4.cbc.pe.dingo
request-cpus-importance-sampling = 64
n-parallel = 4
sampling-requirements = [TARGET.CUDAGlobalMemoryMb>40000]
simple-submission = False

################################################################################
## Sampler / networks
################################################################################
model = {MODEL_PATH}
model-init = {MODEL_INIT_PATH}
device = 'cuda'
num-gnpe-iterations = 30
num-samples = 50_000
batch-size = 10_000
recover-log-prob = true       ; needed for importance sampling

################################################################################
## Data generation
################################################################################
trigger-time = {TIME_EVENT}
label = dingo-SEOBNRv5EHM
outdir = {run_dir}
detectors = [H1, L1]
channel-dict = {{H1:GWOSC, L1:GWOSC}}    
sampling-frequency = 2048
psd-length = 16
post-trigger-duration = 2.0

################################################################################
## Plotting
################################################################################
plot-corner = true
plot-weights = true
plot-log-probs = true
'''

with open(ini_path, "w") as f:
    f.write(ini)
print("wrote", ini_path)

Launch it from a shell (uncomment to run — it builds the DAG and, with `local = True`,
executes generation -> sampling -> importance sampling right here):

In [ ]:
# !dingo_pipe {ini_path}